# ResQ-AI: Full Experiments Notebook

This notebook runs the complete ResQ-AI experiments on Google Colab.

**Requirements:**
- GPU runtime (T4 recommended)
- ~2-4 hours for full experiments

**Change Runtime:**
Runtime → Change runtime type → GPU (T4)

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repo and install
!git clone https://github.com/resq-ai/resq-ai.git 2>/dev/null || true
%cd resq-ai
!pip install -q -r requirements.txt

In [ ]:
# Run smoke test first to verify setup
!python -m pytest tests/test_smoke.py -v -m smoke --tb=short

## Download Real Sen1Floods11 Dataset

Download the official hand-labeled Sentinel-1/2 chips and splits directly from Google Cloud Storage (~3 GB, 1-2 minutes via Colab internal network).

In [ ]:
# Download splits and hand-labeled imagery
!mkdir -p data/sen1floods11/v1.1/splits/flood_handlabeled
!mkdir -p data/sen1floods11/v1.1/data/flood_events/HandLabeled

!gsutil -m cp -r gs://sen1floods11/v1.1/splits/flood_handlabeled/* data/sen1floods11/v1.1/splits/flood_handlabeled/ 2>/dev/null || true
!gsutil -m cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/* data/sen1floods11/v1.1/data/flood_events/HandLabeled/

# Verify download
!ls -la data/sen1floods11/v1.1/data/flood_events/HandLabeled/

## Run Experiments

In [ ]:
# E1: Main model comparison (ResQNet, UNet-SAR, UNet-Optical, EarlyFusion, pixel-level Random Forest across 5 seeds)
!python experiments/run_experiments.py --experiment e1 --config configs/default.yaml

In [ ]:
# E2: Ablation studies (Modality ablation, FiLM/Cross-Attention ablation, Modality Dropout)
!python experiments/run_experiments.py --experiment e2 --config configs/default.yaml

In [ ]:
# E3: Calibration analysis (Deterministic, MC Dropout T=20, TTA, Deep Ensemble M=5)
!python experiments/run_experiments.py --experiment e3 --config configs/default.yaml

In [ ]:
# E4: Conformal prediction (Marginal coverage across alphas and held-out events)
!python experiments/run_experiments.py --experiment e4 --config configs/default.yaml

In [ ]:
# E5: Robustness tests (SAR noise, optical cloud cover, missing modality combinations)
!python experiments/run_experiments.py --experiment e5 --config configs/default.yaml

In [ ]:
# E6: Decision-level allocation evaluation (HiGHS LP solvers, Deterministic, SAA, CVaR, Greedy, Proportional, Oracle, sensitivity analysis)
!python experiments/run_experiments.py --experiment e6 --config configs/default.yaml

In [ ]:
# E7: Efficiency benchmarks (GPU latency, parameter counts, throughput FPS)
!python experiments/run_experiments.py --experiment e7 --config configs/default.yaml

## Generate Figures and Tables

In [ ]:
# Automatically generate all tables and figures directly from aggregated empirical results
!python experiments/generate_figures.py
!python experiments/generate_tables.py

In [ ]:
# Display key figures
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('figures/*.png')):
    print(f'\n--- {fig} ---')
    display(Image(filename=fig, width=600))

## Download Results

In [ ]:
# Zip results for download
!zip -r resq_ai_results.zip results/ figures/ paper/tables/

from google.colab import files
files.download('resq_ai_results.zip')